**This file is focused on the doing of hypotheis testing for the dataset.**

In [40]:
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display


youtube_data = pd.read_csv("../data_github/youtube_public/youtube_activity_public.csv")
spotify_data = pd.read_csv("../data_github/spotify_public/spotify_streaming_public.csv")
netflix_data = pd.read_csv("../data_github/netflix_public/netflix_viewing_public.csv")
prime_data = pd.read_csv("../data_github/prime_video_public/prime_video_watch_history_public.csv")
calendar_data = pd.read_json("../data_github/academical_calendar/academicCalendar.jsonl", lines=True)



ALPHA = 0.05
TIMEZONE = "Europe/Istanbul"

# After-9:30 PM means 21:30-04:59 in Istanbul time.
AFTER_2130_START = 21 * 60 + 30
AFTER_2130_END = 5 * 60

In [41]:
# Making sure that the date columns are in datetime format for all datasets

for data in [youtube_data, spotify_data, netflix_data, prime_data]:
    data["fine_date"] = pd.to_datetime(data["fine_date"])

In [42]:
# Checking the shapes of the datasets to ensure they are loaded correctly
print("Loaded public datasets:")
print("YouTube:", youtube_data.shape)
print("Spotify:", spotify_data.shape)
print("Netflix:", netflix_data.shape)
print("Prime Video:", prime_data.shape)
print("Academic calendar:", calendar_data.shape)

Loaded public datasets:
YouTube: (34317, 16)
Spotify: (178202, 16)
Netflix: (2493, 11)
Prime Video: (719, 17)
Academic calendar: (15, 23)


In [43]:
# YouTube and Spotify have exact timestamps, so they can be used for after-21:30 analysis.
youtube_data["fine_time_start_dt"] = (pd.to_datetime(youtube_data["fine_time_start"], errors="coerce", utc=True).dt.tz_convert(TIMEZONE))

spotify_data["fine_time_start_dt"] = pd.to_datetime(spotify_data["fine_time_start"],errors="coerce",utc=True,format="ISO8601").dt.tz_convert(TIMEZONE)


for data in [youtube_data, spotify_data]:
    data["minute_of_day"] = data["fine_time_start_dt"].dt.hour * 60 + data["fine_time_start_dt"].dt.minute
    data["is_after_2130"] = ((data["minute_of_day"] >= AFTER_2130_START) |(data["minute_of_day"] < AFTER_2130_END))


print("Invalid YouTube timestamps:", youtube_data["fine_time_start_dt"].isna().sum())
print("Invalid Spotify timestamps:", spotify_data["fine_time_start_dt"].isna().sum())


Invalid YouTube timestamps: 0
Invalid Spotify timestamps: 0


In [44]:
common_start_date = max(
    youtube_data["fine_date"].min(),
    spotify_data["fine_date"].min(),
    netflix_data["fine_date"].min(),
    prime_data["fine_date"].min()
)

common_end_date = min(
    youtube_data["fine_date"].max(),
    spotify_data["fine_date"].max(),
    netflix_data["fine_date"].max(),
    prime_data["fine_date"].max()
)

all_dates = pd.DataFrame({
    "fine_date": pd.date_range(common_start_date, common_end_date, freq="D")
})

calendar_rows = []

for _, row in calendar_data.iterrows():
    term_start = pd.to_datetime(row["term_start_date"])
    final_start = pd.to_datetime(row["final_exam_start_date"])
    final_end = pd.to_datetime(row["final_exam_end_date"])

    for date_value in pd.date_range(term_start, final_end, freq="D"):
        if final_start <= date_value <= final_end:
            academic_period = "final_exam"
        else:
            academic_period = "ordinary_term"

        if row["term"] == "summer":
            analysis_period = "summer_work_period"
        else:
            analysis_period = academic_period

        calendar_rows.append({
            "fine_date": date_value,
            "analysis_period": analysis_period
        })

calendar_daily = pd.DataFrame(calendar_rows).drop_duplicates("fine_date", keep="first")

print("Common date range:")
print(common_start_date, "to", common_end_date)
print("Number of dates:", len(all_dates))


Common date range:
2022-04-17 00:00:00 to 2026-03-10 00:00:00
Number of dates: 1424


In [45]:
youtube_daily = (
    youtube_data
    .groupby("fine_date")
    .agg(
        youtube_daily_total_count=("fine_record_id", "count"),
        youtube_daily_watched_count=("action", lambda x: (x == "Watched").sum()),
        youtube_daily_search_count=("action", lambda x: (x == "Searched for").sum()),
        youtube_after_2130_count=("is_after_2130", "sum")
    )
    .reset_index()
)


youtube_after_2130_watched = (
    youtube_data[
        (youtube_data["is_after_2130"]) &
        (youtube_data["action"] == "Watched")
    ]
    .groupby("fine_date")
    .size()
    .reset_index(name="youtube_after_2130_watched_count")
)

youtube_daily = youtube_daily.merge(youtube_after_2130_watched, on="fine_date", how="left")

youtube_daily["youtube_after_2130_watched_count"] = (
    youtube_daily["youtube_after_2130_watched_count"]
    .fillna(0)
    .astype(int)
)


In [46]:
youtube_daily

,fine_date,youtube_daily_total_count,youtube_daily_watched_count,youtube_daily_search_count,youtube_after_2130_count,youtube_after_2130_watched_count
0,2022-04-17,1,0,0,0,0
1,2022-05-30,1,0,0,1,0
2,2022-06-05,1,0,0,0,0
3,2022-06-21,2,0,0,1,0
4,2022-10-22,1,0,0,0,0
...,...,...,...,...,...,...
850,2026-03-11,50,47,2,0,0
851,2026-03-12,86,84,1,0,0
852,2026-03-13,19,13,6,16,11
853,2026-03-14,30,27,2,2,1


In [47]:
spotify_daily = (
    spotify_data
    .groupby("fine_date")
    .agg(
        spotify_daily_hours=("hours_played", "sum"),
        spotify_daily_stream_count=("fine_record_id", "count")
    )
    .reset_index()
)

spotify_after_2130 = (
    spotify_data[spotify_data["is_after_2130"]]
    .groupby("fine_date")
    .agg(
        spotify_after_2130_hours=("hours_played", "sum"),
        spotify_after_2130_stream_count=("fine_record_id", "count")
    )
    .reset_index()
)

spotify_daily = spotify_daily.merge(spotify_after_2130, on="fine_date", how="left")


spotify_daily["spotify_after_2130_hours"] = (
    spotify_daily["spotify_after_2130_hours"]
    .fillna(0)
)

spotify_daily["spotify_after_2130_stream_count"] = (
    spotify_daily["spotify_after_2130_stream_count"]
    .fillna(0)
    .astype(int)
)



In [48]:
spotify_daily

,fine_date,spotify_daily_hours,spotify_daily_stream_count,spotify_after_2130_hours,spotify_after_2130_stream_count
0,2019-07-27,0.314114,5,0.000000,0
1,2019-07-28,0.861175,20,0.000000,0
2,2019-07-29,1.745592,34,1.109861,24
3,2019-07-30,0.530142,20,0.530142,20
4,2019-07-31,0.653267,13,0.000000,0
...,...,...,...,...,...
2127,2026-03-10,3.768885,161,0.037794,1
2128,2026-03-11,2.113020,61,0.083945,2
2129,2026-03-12,0.182897,14,0.004516,1
2130,2026-03-13,1.160088,171,0.000000,0


In [49]:
netflix_daily = (
    netflix_data
    .groupby("fine_date")
    .size()
    .reset_index(name="netflix_daily_count")
)

prime_daily = (
    prime_data
    .groupby("fine_date")
    .size()
    .reset_index(name="prime_video_daily_count")
)

# This prevents the datetime/object merge error.
for df in [all_dates, calendar_daily, youtube_daily, spotify_daily, netflix_daily, prime_daily]:
    df["fine_date"] = pd.to_datetime(df["fine_date"])


In [50]:
netflix_daily

,fine_date,netflix_daily_count
0,2020-02-25,1
1,2020-03-20,1
2,2020-04-15,1
3,2020-04-18,1
4,2020-04-19,8
...,...,...
321,2026-01-18,17
322,2026-01-20,2
323,2026-01-27,1
324,2026-02-07,1


In [51]:
prime_daily

,fine_date,prime_video_daily_count
0,2022-03-30,2
1,2022-04-03,3
2,2022-04-13,14
3,2022-04-14,1
4,2022-04-30,1
...,...,...
95,2026-03-14,22
96,2026-03-15,8
97,2026-03-16,12
98,2026-03-17,2


In [52]:
combined_daily = (
    all_dates
    .merge(youtube_daily, on="fine_date", how="left")
    .merge(spotify_daily, on="fine_date", how="left")
    .merge(netflix_daily, on="fine_date", how="left")
    .merge(prime_daily, on="fine_date", how="left")
    .merge(calendar_daily, on="fine_date", how="left")
)

combined_daily["analysis_period"] = combined_daily["analysis_period"].fillna("outside_calendar")

activity_columns = [
    "youtube_daily_total_count",
    "youtube_daily_watched_count",
    "youtube_daily_search_count",
    "youtube_after_2130_count",
    "youtube_after_2130_watched_count",
    "spotify_daily_hours",
    "spotify_daily_stream_count",
    "spotify_after_2130_hours",
    "spotify_after_2130_stream_count",
    "netflix_daily_count",
    "prime_video_daily_count"
]

combined_daily[activity_columns] = combined_daily[activity_columns].fillna(0)

count_columns = [
    "youtube_daily_total_count",
    "youtube_daily_watched_count",
    "youtube_daily_search_count",
    "youtube_after_2130_count",
    "youtube_after_2130_watched_count",
    "spotify_daily_stream_count",
    "spotify_after_2130_stream_count",
    "netflix_daily_count",
    "prime_video_daily_count"
]

for column in count_columns:
    combined_daily[column] = combined_daily[column].astype(int)

combined_daily["netflix_prime_daily_count"] = (
    combined_daily["netflix_daily_count"] +
    combined_daily["prime_video_daily_count"]
)

combined_daily["netflix_prime_active_day"] = combined_daily["netflix_prime_daily_count"] > 0

combined_daily["daily_distinct_entertainment_platform_count"] = (
    (combined_daily["youtube_daily_total_count"] > 0).astype(int) +
    (combined_daily["spotify_daily_stream_count"] > 0).astype(int) +
    (combined_daily["netflix_daily_count"] > 0).astype(int) +
    (combined_daily["prime_video_daily_count"] > 0).astype(int)
)

combined_daily["youtube_after_2130_share"] = (
    combined_daily["youtube_after_2130_count"] /
    combined_daily["youtube_daily_total_count"].replace(0, np.nan)
).fillna(0)

combined_daily["spotify_after_2130_hour_share"] = (
    combined_daily["spotify_after_2130_hours"] /
    combined_daily["spotify_daily_hours"].replace(0, np.nan)
).fillna(0)

print("Combined daily shape:", combined_daily.shape)
print(combined_daily["analysis_period"].value_counts())
display(combined_daily.head())


Combined daily shape: (1424, 18)
analysis_period
ordinary_term         775
outside_calendar      348
summer_work_period    204
final_exam             97
Name: count, dtype: int64


,fine_date,youtube_daily_total_count,youtube_daily_watched_count,youtube_daily_search_count,youtube_after_2130_count,youtube_after_2130_watched_count,spotify_daily_hours,spotify_daily_stream_count,spotify_after_2130_hours,spotify_after_2130_stream_count,netflix_daily_count,prime_video_daily_count,analysis_period,netflix_prime_daily_count,netflix_prime_active_day,daily_distinct_entertainment_platform_count,youtube_after_2130_share,spotify_after_2130_hour_share
0,2022-04-17,1,0,0,0,0,1.603053,70,0.117174,25,0,0,ordinary_term,0,False,2,0.0,0.073094
1,2022-04-18,0,0,0,0,0,0.385654,11,0.264308,10,0,0,ordinary_term,0,False,1,0.0,0.685350
2,2022-04-19,0,0,0,0,0,0.002694,7,0.000000,1,0,0,ordinary_term,0,False,1,0.0,0.000000
3,2022-04-20,0,0,0,0,0,0.905688,135,0.905688,135,1,0,ordinary_term,1,True,2,0.0,1.000000
4,2022-04-21,0,0,0,0,0,0.634982,64,0.268858,28,0,0,ordinary_term,0,False,1,0.0,0.423410


In [53]:
assert common_start_date == pd.Timestamp("2022-04-17")
assert common_end_date == pd.Timestamp("2026-03-10")
assert len(combined_daily) == 1424
assert combined_daily["fine_date"].is_unique

assert (
    combined_daily["netflix_prime_daily_count"] ==
    combined_daily["netflix_daily_count"] + combined_daily["prime_video_daily_count"]
).all()

print("Validation checks passed.")


Validation checks passed.


In [54]:
results = []

def decision_text(p_value):
    if p_value < ALPHA:
        return "Reject H0"
    return "Do not reject H0"


def mann_whitney_less(hypothesis, outcome, group_1_label, group_1_values, group_2_label, group_2_values):
    result = stats.mannwhitneyu(group_1_values, group_2_values, alternative="less")

    results.append({
        "hypothesis": hypothesis,
        "outcome": outcome,
        "test": "Mann-Whitney U, one-sided less",
        "comparison": f"{group_1_label} < {group_2_label}",
        "n_group_1": len(group_1_values),
        "n_group_2": len(group_2_values),
        "mean_group_1": group_1_values.mean(),
        "mean_group_2": group_2_values.mean(),
        "median_group_1": group_1_values.median(),
        "median_group_2": group_2_values.median(),
        "statistic": result.statistic,
        "raw_p_value": result.pvalue,
        "alpha": ALPHA,
        "decision": decision_text(result.pvalue)
    })


def spearman_greater(hypothesis, outcome, x_column, y_column):
    result = stats.spearmanr(
        combined_daily[x_column],
        combined_daily[y_column],
        alternative="greater"
    )

    results.append({
        "hypothesis": hypothesis,
        "outcome": outcome,
        "test": "Spearman correlation, one-sided greater",
        "comparison": f"{x_column} positively associated with {y_column}",
        "n_group_1": len(combined_daily),
        "n_group_2": len(combined_daily),
        "mean_group_1": combined_daily[x_column].mean(),
        "mean_group_2": combined_daily[y_column].mean(),
        "median_group_1": combined_daily[x_column].median(),
        "median_group_2": combined_daily[y_column].median(),
        "statistic": result.statistic,
        "raw_p_value": result.pvalue,
        "alpha": ALPHA,
        "decision": decision_text(result.pvalue)
    })


In [55]:
test_columns = [
    "youtube_daily_total_count",
    "youtube_daily_watched_count",
    "youtube_daily_search_count",
    "youtube_after_2130_count",
    "youtube_after_2130_watched_count",
    "spotify_daily_hours",
    "spotify_daily_stream_count",
    "spotify_after_2130_hours",
    "spotify_after_2130_stream_count",
    "netflix_daily_count",
    "prime_video_daily_count",
    "netflix_prime_daily_count",
    "daily_distinct_entertainment_platform_count",
    "youtube_after_2130_share",
    "spotify_after_2130_hour_share"
]

combined_daily[test_columns].isna().sum()


youtube_daily_total_count                      0
youtube_daily_watched_count                    0
youtube_daily_search_count                     0
youtube_after_2130_count                       0
youtube_after_2130_watched_count               0
spotify_daily_hours                            0
spotify_daily_stream_count                     0
spotify_after_2130_hours                       0
spotify_after_2130_stream_count                0
netflix_daily_count                            0
prime_video_daily_count                        0
netflix_prime_daily_count                      0
daily_distinct_entertainment_platform_count    0
youtube_after_2130_share                       0
spotify_after_2130_hour_share                  0
dtype: int64

In [56]:
final_days = combined_daily[combined_daily["analysis_period"] == "final_exam"]
ordinary_days = combined_daily[combined_daily["analysis_period"] == "ordinary_term"]

active_long_form_days = combined_daily[combined_daily["netflix_prime_active_day"]]
inactive_long_form_days = combined_daily[~combined_daily["netflix_prime_active_day"]]

# H1: Entertainment usage during academic pressure.
h1_outcomes = [
    ("youtube_daily_watched_count", "YouTube watched count"),
    ("spotify_daily_hours", "Spotify listening hours"),
    ("netflix_daily_count", "Netflix viewing count"),
    ("prime_video_daily_count", "Prime Video viewing count")
]

for column, label in h1_outcomes:
    mann_whitney_less(
        "H1 Entertainment usage during academic pressure",
        label,
        "final_exam",
        final_days[column],
        "ordinary_term",
        ordinary_days[column]
    )

# H2: Platform diversity during academic pressure.
mann_whitney_less(
    "H2 Platform diversity during academic pressure",
    "Daily distinct entertainment platform count",
    "final_exam",
    final_days["daily_distinct_entertainment_platform_count"],
    "ordinary_term",
    ordinary_days["daily_distinct_entertainment_platform_count"]
)

# H3: Long-form streaming and YouTube activity.
mann_whitney_less(
    "H3 Long-form streaming and YouTube activity",
    "YouTube watched count",
    "Netflix+Prime active days",
    active_long_form_days["youtube_daily_watched_count"],
    "Netflix+Prime inactive days",
    inactive_long_form_days["youtube_daily_watched_count"]
)

# H4: Spotify and YouTube co-usage.
spearman_greater(
    "H4 Spotify and YouTube co-usage",
    "Spotify hours and YouTube watched count",
    "spotify_daily_hours",
    "youtube_daily_watched_count"
)

# H5: After-9:30 PM entertainment during finals.
mann_whitney_less(
    "H5 After-9:30 PM entertainment during finals",
    "YouTube after-21:30 activity share",
    "final_exam",
    final_days["youtube_after_2130_share"],
    "ordinary_term",
    ordinary_days["youtube_after_2130_share"]
)

mann_whitney_less(
    "H5 After-9:30 PM entertainment during finals",
    "Spotify after-21:30 listening-hour share",
    "final_exam",
    final_days["spotify_after_2130_hour_share"],
    "ordinary_term",
    ordinary_days["spotify_after_2130_hour_share"]
)


In [57]:
hypothesis_results = pd.DataFrame(results)

round_columns = [
    "mean_group_1",
    "mean_group_2",
    "median_group_1",
    "median_group_2",
    "statistic",
    "raw_p_value"
]

hypothesis_results[round_columns] = hypothesis_results[round_columns].round(4)

display(hypothesis_results)


,hypothesis,outcome,test,comparison,n_group_1,n_group_2,mean_group_1,mean_group_2,median_group_1,median_group_2,statistic,raw_p_value,alpha,decision
0,H1 Entertainment usage during academic pressure,YouTube watched count,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,24.3505,30.7639,8.0000,12.0000,34501.0000,0.0908,0.05,Do not reject H0
1,H1 Entertainment usage during academic pressure,Spotify listening hours,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,1.6459,1.9106,1.4193,1.6417,34130.0000,0.0697,0.05,Do not reject H0
2,H1 Entertainment usage during academic pressure,Netflix viewing count,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,1.0206,0.8994,0.0000,0.0000,38928.5000,0.8512,0.05,Do not reject H0
3,H1 Entertainment usage during academic pressure,Prime Video viewing count,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,0.2165,0.5587,0.0000,0.0000,37300.0000,0.3890,0.05,Do not reject H0
4,H2 Platform diversity during academic pressure,Daily distinct entertainment platform count,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,1.7938,1.8697,2.0000,2.0000,35360.0000,0.1305,0.05,Do not reject H0
5,H3 Long-form streaming and YouTube activity,YouTube watched count,"Mann-Whitney U, one-sided less",Netflix+Prime active days < Netflix+Prime inac...,264,1160,26.4129,21.3672,4.0000,3.0000,159759.5000,0.8731,0.05,Do not reject H0
6,H4 Spotify and YouTube co-usage,Spotify hours and YouTube watched count,"Spearman correlation, one-sided greater",spotify_daily_hours positively associated with...,1424,1424,1.8890,22.3027,1.5663,3.0000,0.0934,0.0002,0.05,Reject H0
7,H5 After-9:30 PM entertainment during finals,YouTube after-21:30 activity share,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,0.1786,0.2732,0.0000,0.0000,32124.0000,0.0059,0.05,Reject H0
8,H5 After-9:30 PM entertainment during finals,Spotify after-21:30 listening-hour share,"Mann-Whitney U, one-sided less",final_exam < ordinary_term,97,775,0.1791,0.2195,0.0140,0.0377,33423.0000,0.0342,0.05,Reject H0


In [58]:
period_context_summary = (
    combined_daily
    .groupby("analysis_period")[[
        "youtube_daily_watched_count",
        "spotify_daily_hours",
        "netflix_daily_count",
        "prime_video_daily_count",
        "netflix_prime_daily_count",
        "daily_distinct_entertainment_platform_count",
        "youtube_after_2130_share",
        "spotify_after_2130_hour_share"
    ]]
    .mean()
    .loc[["ordinary_term", "final_exam", "outside_calendar", "summer_work_period"]]
    .round(4)
)

display(period_context_summary)


,youtube_daily_watched_count,spotify_daily_hours,netflix_daily_count,prime_video_daily_count,netflix_prime_daily_count,daily_distinct_entertainment_platform_count,youtube_after_2130_share,spotify_after_2130_hour_share
analysis_period,,,,,,,,
ordinary_term,30.7639,1.9106,0.8994,0.5587,1.4581,1.8697,0.2732,0.2195
final_exam,24.3505,1.6459,1.0206,0.2165,1.2371,1.7938,0.1786,0.1791
outside_calendar,13.2615,1.7739,1.4138,0.3563,1.7701,1.6207,0.1639,0.1791
summer_work_period,4.6078,2.1186,0.7990,0.2353,1.0343,1.4853,0.1100,0.1588
